# Custocare — Subscription Lifecycle & State Machine

**Internal Architecture Document** · May 2026

---


## 1. Subscription Fields

Each facility has exactly **one** subscription record. Status transitions update the same record — no new records are created.

| Field | Type | Purpose |
|-------|------|---------|
| `status` | enum | `trial`, `active`, `past_due`, `suspended`, `cancelled` |
| `plan_id` | FK → plans | Current plan |
| `billing_cycle` | `monthly` / `yearly` | Billing frequency |
| `starts_at` | timestamp | When current period started |
| `ends_at` | timestamp | End of current billing period |
| `next_billing_date` | timestamp | Primary driver for payment-due transitions |
| `trial_ends_at` | timestamp | **Set once, never cleared** |
| `grace_period_ends_at` | timestamp | **Set once, never cleared** |
| `suspended_at` | timestamp | When suspended |
| `approved_at` | timestamp | When payment was last approved by admin |
| `onboarding_fee_paid` | bool | Whether onboarding fee was paid |
| `metadata` | JSON | `cancel_at_period_end`, `pending_upgrade_plan_id`, `access_ends_at` |

---


## 2. Immutable Markers

Two fields, once set, are **never overwritten with null** by any code path:

- **`trial_ends_at`** — set when a trial is first granted. Remains after payment, plan switches, suspension, cancellation. Used by `hasEverHadTrial()`.
- **`grace_period_ends_at`** — set when subscription first goes past_due with active grace. Remains after payment, suspension, resubscription.

Both are only included in update payloads when their value is non-null.

---
## 3. Transition Triggers

Each transition is driven by a single field compared against `now()`:

| Transition | Trigger Field | Check |
|-----------|--------------|-------|
| trial → past_due | `next_billing_date` **or** `trial_ends_at` | `isPast()` |
| active → past_due | `next_billing_date` | `isPast()` |
| past_due → suspended | `grace_period_ends_at` | `isPast()` |
| cancel_at_period_end → suspended | `ends_at` | `isPast()` (only when `cancel_at_period_end` is true) |
| suspended/any → active | Admin approves payment | Manual — no date check |

### Field Roles

- **`next_billing_date`** — Primary driver. Set to `trial_end` for trial, `period_end` for active.
- **`grace_period_ends_at`** — Sole driver for past_due → suspended.
- **`trial_ends_at`** — Secondary safety net for trial expiry.
- **`ends_at`** — Only checked when `cancel_at_period_end` is true.

### Order of Checks

```python
# In SubscriptionService.getSubscriptionForFacility()
1. if status in (active, trial) and next_billing_date is past → markPastDue()
2. if status is past_due and grace_period_ends_at is past → suspendSubscription()
3. apply pending scheduled changes
```

---


In [ ]:
# Status values
STATUS = ['trial', 'active', 'past_due', 'suspended', 'cancelled']
print('Valid subscription statuses:', STATUS)

---
## 4. Transition Map

### 4.1 First Subscribe

```
status:        null → trial
starts_at:     now
ends_at:       trial_end + billing_cycle
next_billing:  trial_end
trial_ends_at: now + plan.trial_days  ← SET ONCE
grace:         null
```

**Trigger:** `POST /subscription` → `createSubscription()`

---
### 4.2 Switch Plans Mid-Trial

```
plan_id:       new plan
billing_cycle: new cycle
trial_ends_at: PRESERVED (unchanged)
```

Only `plan_id` and `billing_cycle` update. Trial clock keeps ticking — no reset.

---
### 4.3 Trial Expires (auto)

```
trial → past_due
grace: now + 7d  ← SET ONCE (only if null)
trial_ends_at: PRESERVED
```

---
### 4.4 Payment Approved (admin)

```
any → active
starts_at:     now
ends_at:       now + billing_cycle
next_billing:  now + billing_cycle
approved_at:   now
suspended_at:  null
trial_ends_at: PRESERVED
grace:         PRESERVED
```

---
### 4.5 Billing Date Passes (auto)

```
active → past_due
grace: now + 7d  (only if null)
```

---
### 4.6 Grace Expires (auto)

```
past_due → suspended
suspended_at: now
grace: PRESERVED
```

---
### 4.7 Resubscribe While Suspended

```
plan_id:       new plan
billing_cycle: new cycle
status:        SUSPENDED (stays suspended)
trial_ends_at: PRESERVED
grace:         PRESERVED
```

No trial, no grace. Payment is the only way out.

---
### 4.8 Cancel at Period End

```
metadata.cancel_at_period_end: true
status: active (until ends_at)
When ends_at passes → auto-suspend
```

---


In [ ]:
# Transition simulation (conceptual)
from datetime import datetime, timedelta

def transition(trigger_field, now, value):
    """Check if a transition should fire."""
    return value is not None and value < now

now = datetime.utcnow()
print(f'Current time: {now}')
print(f'Trial ends at (past): {now - timedelta(days=1)} → expired: {transition("trial_ends_at", now, now - timedelta(days=1))}')
print(f'Next billing (future): {now + timedelta(days=30)} → due: {transition("next_billing_date", now, now + timedelta(days=30))}')

---
## 5. Frontend Payment Action Resolver

The backend determines `payment_action` based on subscription state:

| Status | Condition | `required` | `intent` |
|--------|-----------|-----------|----------|
| Any | Pending payment exists | `false` | — |
| Any | `pending_upgrade_plan_id` in metadata | `true` | `upgrade_now` |
| Trial | Not approved, trial active | `true` | `subscription` |
| Past_due | — | `true` | `renewal` |
| Active | — | `false` | — |
| Suspended | — | `false` | — |
| Cancelled | — | `false` | — |

The frontend `resolvePaymentQuoteParams()` falls back to `subscription` intent for the current plan when `payment_action` has no specific intent.

---


In [ ]:
# Payment action by status
payment_actions = {
    'trial':     {'required': True,  'intent': 'subscription'},
    'past_due':  {'required': True,  'intent': 'renewal'},
    'active':    {'required': False, 'intent': None},
    'suspended': {'required': False, 'intent': None},
    'cancelled': {'required': False, 'intent': None},
}

for status, action in payment_actions.items():
    print(f"  {status:12s} → required={action['required']}, intent={action['intent']}")

---
## 6. One Subscription Per Facility

Design rule: **one subscription record per facility**. Statuses change, fields update, but the same record persists.

- No new subscription on resubscribe — existing record is updated
- `trial_ends_at` and `grace_period_ends_at` are set once and never cleared
- `hasEverHadTrial()` searches all records (including soft-deleted) to prevent second trials
- Grace is granted once per facility lifetime, detected via `grace_period_ends_at !== null`

### Anti-Abuse Summary

| Scenario | Gets trial? | Gets grace? |
|----------|-----------|------------|
| First subscribe | ✅ 14 days | — |
| Trial expires | — | ✅ 7 days (once) |
| Grace expires → suspended | — | ❌ |
| Resubscribe while suspended | ❌ (`hasEverHadTrial`) | ❌ (`hasUsedGraceBefore`) |
| Pay → active → expires again | ❌ | ❌ |

---


In [ ]:
# Anti-abuse simulation
def get_remaining_trial_days(has_ever_had_trial, plan_trial_days):
    return 0 if has_ever_had_trial else plan_trial_days

def get_grace(has_used_grace_before):
    return None if has_used_grace_before else 7

scenarios = [
    ('First subscribe',        False, False, 14),
    ('Trial expired',           True,  False, 14),
    ('Grace expired',           True,  True,  14),
    ('Resubscribe suspended',   True,  True,  14),
    ('Pay → active → expires', True,  True,  14),
]

print(f"{'Scenario':30s} {'Trial days':12s} {'Grace':8s}")
print('-' * 50)
for name, has_trial, has_grace, plan_days in scenarios:
    trial = get_remaining_trial_days(has_trial, plan_days)
    grace = get_grace(has_grace)
    print(f"{name:30s} {str(trial):12s} {str(grace):8s}")

---
## 7. Key Code Locations

| Component | File |
|-----------|------|
| Subscription creation/update | `app/Services/Billing/SubscriptionService.php` |
| Auto-transition logic | `SubscriptionService::getSubscriptionForFacility()` |
| Grace expiry → suspend | `SubscriptionService::markPastDue()`, `suspendSubscription()` |
| Payment action resolver | `app/Services/Billing/SubscriptionPaymentActionResolver.php` |
| Scheduled change application | `app/Services/Billing/SubscriptionScheduledChangeService.php` |
| Middleware (blocks suspended) | `app/Http/Middleware/EnsureFacilitySubscriptionIsActive.php` |
| Repository (findByFacility) | `app/Repositories/Billing/SubscriptionRepository.php` |
| Frontend Payment utils | `src/.../subscriptionPaymentUtils.ts` |
| Frontend Navbar display | `src/.../Navbar/Subscription.tsx` |
| Frontend Plan selection | `src/.../AvailablePlans.tsx` |
| Frontend Payment page | `src/.../Payments.tsx` |

---

*End of document*